<!-- MUTERBANDUNG_NOTEBOOK_STATUS_V1 -->
# Notebook Status - Colab Model Berat

| Item | Isi |
|---|---|
| Status | COLAB_MODEL_BERAT |
| Fungsi | Eksperimen/training model sentiment berbasis transformer. |
| Input utama | Dataset review yang sudah bersih dan siap training. |
| Output utama | Model/artifact evaluasi jika training dijalankan. |
| Aturan pakai | Jalankan di Colab/GPU. Jangan dipakai sebagai bukti model aktif sebelum artifact dan evaluasi diverifikasi. |
| Catatan | Untuk runtime sistem, sentiment sebaiknya tetap precomputed/offline. |

---


# 🧠 MuterBandung AI - IndoBERT Sentiment Classification
Notebook ini didesain **KHUSUS UNTUK GOOGLE COLAB** karena menggunakan *Deep Learning* (PyTorch & Transformers) yang membutuhkan akselerasi GPU.

**Instruksi Persiapan:**
1. Klik menu `Runtime` > `Change runtime type` > Pilih Hardware accelerator: **T4 GPU**
2. Upload file `MASTER_REVIEWS_NLP.csv` ke dalam folder `/content/` (menu Files di sebelah kiri)
3. Jalankan semua cell secara berurutan.

In [ ]:
# 1. INSTALL DEPENDENSI
!pip install transformers torch pandas scikit-learn tqdm -q

import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time

# Pastikan GPU aktif
device = 0 if torch.cuda.is_available() else -1
print("GPU Aktif:" if device == 0 else "Hanya CPU aktif. HARAP GANTI RUNTIME KE GPU!")

In [ ]:
# 2. LOAD DATASET
print("Memuat dataset...")
df = pd.read_csv('MASTER_REVIEWS_NLP.csv')
df = df.dropna(subset=['review_nlp'])

def label_sentimen(rating):
    if rating >= 4: return 'positif'
    elif rating == 3: return 'netral'
    else: return 'negatif'

df['sentimen'] = df['rating'].apply(label_sentimen)

# Menggunakan sampel jika data terlalu besar (misal 5000 ulasan) 
# Hapus .sample() jika ingin melatih ke seluruh 15.000 data
df_run = df.sample(5000, random_state=42) if len(df) > 5000 else df
print(f"Total data diproses: {len(df_run)}")

In [ ]:
# 3. INISIALISASI INDOBERT PIPELINE
print("Mendownload Model IndoBERT dari HuggingFace...")
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="mdhugol/indonesia-bert-sentiment-classification", 
    tokenizer="mdhugol/indonesia-bert-sentiment-classification",
    device=device
)

def map_indobert_label(label):
    if label == 'LABEL_0': return 'positif'
    elif label == 'LABEL_1': return 'netral'
    elif label == 'LABEL_2': return 'negatif'
    return 'netral'

# 4. INFERENSI (PREDIKSI)
print("Mulai memprediksi sentimen (membutuhkan waktu beberapa menit)...")
start = time.time()
teks_list = df_run['review_text_clean'].astype(str).tolist()

# Batasi panjang karakter untuk mencegah out of memory
teks_list = [t[:512] for t in teks_list] 

predictions = sentiment_pipeline(teks_list, truncation=True, max_length=512)
elapsed = time.time() - start
print(f"Prediksi selesai dalam {elapsed:.1f} detik.")

df_run['sentimen_prediksi'] = [map_indobert_label(p['label']) for p in predictions]

In [ ]:
# 5. EVALUASI MODEL
y_true = df_run['sentimen']
y_pred = df_run['sentimen_prediksi']

print("="*50)
print("HASIL EVALUASI INDOBERT (3 LABEL)")
print("="*50)
print(f"Akurasi Keseluruhan: {accuracy_score(y_true, y_pred)*100:.2f}%\n")
print(classification_report(y_true, y_pred, target_names=['negatif', 'netral', 'positif']))

# Simpan hasil akhir
df_run.to_csv('INDOBERT_LABELED_RESULTS.csv', index=False)
print("Hasil tersimpan di INDOBERT_LABELED_RESULTS.csv")